In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization, LSTM, Bidirectional, Masking
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasRegressor
from scipy.stats import loguniform, randint
from sklearn.base import BaseEstimator, TransformerMixin
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load the dataset
data_path = '/LR!_datasets/V5.csv'
df = pd.read_csv(data_path)

# Display basic information about the dataset
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())
print("\nFirst few rows:")
print(df.head())

# Data Preparation
# Remove motor_UPDRS as instructed
df = df.drop('motor_UPDRS', axis=1)

# EDA: Basic statistics
print("\nDataset statistics:")
print(df.describe())

# EDA: Visualize distributions
plt.figure(figsize=(12, 8))
df['total_UPDRS'].hist(bins=30)
plt.title('Distribution of total_UPDRS')
plt.xlabel('total_UPDRS')
plt.ylabel('Frequency')
plt.show()

# EDA: Correlation matrix
corr_matrix = df.corr()
plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# EDA: Time series visualization for a few subjects
subject_sample = np.random.choice(df['subject#'].unique(), 4, replace=False)
plt.figure(figsize=(12, 8))
for i, subject in enumerate(subject_sample):
    subject_data = df[df['subject#'] == subject].sort_values('test_time')
    plt.subplot(2, 2, i+1)
    plt.plot(subject_data['test_time'], subject_data['total_UPDRS'], marker='o')
    plt.title(f'Subject {subject}')
    plt.xlabel('Test Time')
    plt.ylabel('total_UPDRS')
plt.tight_layout()
plt.show()

# Data preprocessing functions
class SequenceBuilder(BaseEstimator, TransformerMixin):
    """Transforms tabular data into sequences for RNNs"""
    def __init__(self, sequence_length=5, group_col='subject#', sort_col='test_time', target_col='total_UPDRS'):
        self.sequence_length = sequence_length
        self.group_col = group_col
        self.sort_col = sort_col
        self.target_col = target_col
        self.feature_names = None

    def fit(self, X, y=None):
        if hasattr(X, 'columns'):
            self.feature_names = X.columns.drop(
                [self.group_col, self.sort_col, self.target_col], errors='ignore').tolist()
        return self

    def transform(self, X, y=None):
        if hasattr(X, 'columns'):
            feature_df = X[self.feature_names].copy()
            groups = X[self.group_col].values
            times = X[self.sort_col].values
            targets = X[self.target_col].values if self.target_col in X else y
        else:
            raise ValueError("X must be a DataFrame with columns")
            
        unique_groups = np.unique(groups)
        X_seq = []
        y_seq = []
        group_ids = []
        
        for group in unique_groups:
            group_mask = groups == group
            group_features = feature_df[group_mask].values
            group_times = times[group_mask]
            group_targets = targets[group_mask]
            
            # Sort by time
            sort_idx = np.argsort(group_times)
            group_features = group_features[sort_idx]
            group_targets = group_targets[sort_idx]
            
            # Create sequences
            for i in range(len(group_features) - self.sequence_length + 1):
                X_seq.append(group_features[i:i+self.sequence_length])
                y_seq.append(group_targets[i+self.sequence_length-1])
                group_ids.append(group)
        
        # Handle cases where we don't have enough sequences
        if len(X_seq) == 0:
            # Pad sequences for subjects with insufficient data
            for group in unique_groups:
                group_mask = groups == group
                group_features = feature_df[group_mask].values
                group_times = times[group_mask]
                group_targets = targets[group_mask]
                
                # Sort by time
                sort_idx = np.argsort(group_times)
                group_features = group_features[sort_idx]
                group_targets = group_targets[sort_idx]
                
                if len(group_features) > 0:
                    # Pad with zeros to reach sequence_length
                    padding = np.zeros((self.sequence_length - len(group_features), group_features.shape[1]))
                    padded_features = np.vstack([padding, group_features])
                    X_seq.append(padded_features)
                    y_seq.append(group_targets[-1])
                    group_ids.append(group)
        
        return np.array(X_seq), np.array(y_seq), np.array(group_ids)

# Prepare the data
# Separate features and target
X = df.drop('total_UPDRS', axis=1)
y = df['total_UPDRS']

# Split by subject to avoid data leakage
subjects = X['subject#'].unique()
train_subjects, test_subjects = train_test_split(subjects, test_size=0.15, random_state=42)
train_subjects, val_subjects = train_test_split(train_subjects, test_size=0.176, random_state=42)  # 15% of original

# Create masks for train, validation, and test sets
train_mask = X['subject#'].isin(train_subjects)
val_mask = X['subject#'].isin(val_subjects)
test_mask = X['subject#'].isin(test_subjects)

X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]
y_train, y_val, y_test = y[train_mask], y[val_mask], y[test_mask]

# Impute missing values
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

# Convert back to DataFrames
X_train_imp = pd.DataFrame(X_train_imputed, columns=X_train.columns, index=X_train.index)
X_val_imp = pd.DataFrame(X_val_imputed, columns=X_val.columns, index=X_val.index)
X_test_imp = pd.DataFrame(X_test_imputed, columns=X_test.columns, index=X_test.index)

# Feature engineering
def create_features(df):
    """Create additional features through engineering"""
    df = df.copy()
    
    # Create time since first measurement for each subject
    df['time_since_first'] = df.groupby('subject#')['test_time'].transform(
        lambda x: x - x.min())
    
    # Create rolling statistics for key features (per subject)
    acoustic_features = [col for col in df.columns if 'Jitter' in col or 'Shimmer' in col 
                         or 'NHR' in col or 'HNR' in col or 'RPDE' in col or 'PPE' in col]
    
    for feature in acoustic_features:
        df[f'{feature}_rolling_mean'] = df.groupby('subject#')[feature].transform(
            lambda x: x.rolling(3, min_periods=1).mean())
        df[f'{feature}_rolling_std'] = df.groupby('subject#')[feature].transform(
            lambda x: x.rolling(3, min_periods=1).std())
    
    # Create ratio features
    if 'Jitter(%)' in df.columns and 'Shimmer(%)' in df.columns:
        df['Jitter_Shimmer_ratio'] = df['Jitter(%)'] / (df['Shimmer(%)'] + 1e-6)
    
    return df

# Apply feature engineering
X_train_fe = create_features(X_train_imp)
X_val_fe = create_features(X_val_imp)
X_test_fe = create_features(X_test_imp)

# Feature selection
# Select top features based on correlation with target
correlation_with_target = X_train_fe.corrwith(y_train).abs().sort_values(ascending=False)
top_features = correlation_with_target.head(15).index.tolist()

X_train_fs = X_train_fe[top_features]
X_val_fs = X_val_fe[top_features]
X_test_fs = X_test_fe[top_features]

# Standardization
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train_fs)
X_val_std = scaler.transform(X_val_fs)
X_test_std = scaler.transform(X_test_fs)

# Convert back to DataFrames
X_train_std_df = pd.DataFrame(X_train_std, columns=X_train_fs.columns, index=X_train_fs.index)
X_val_std_df = pd.DataFrame(X_val_std, columns=X_val_fs.columns, index=X_val_fs.index)
X_test_std_df = pd.DataFrame(X_test_std, columns=X_test_fs.columns, index=X_test_fs.index)

# Prepare the four dataset variants
# 1. Original dataset (imputed)
X_train_orig = X_train_imp
X_val_orig = X_val_imp
X_test_orig = X_test_imp

# 2. Transformed original (standardized)
scaler_orig = StandardScaler()
X_train_orig_std = scaler_orig.fit_transform(X_train_orig)
X_val_orig_std = scaler_orig.transform(X_val_orig)
X_test_orig_std = scaler_orig.transform(X_test_orig)

# 3. Constructed dataset (feature engineered and selected)
X_train_const = X_train_fs
X_val_const = X_val_fs
X_test_const = X_test_fs

# 4. Transformed constructed dataset (standardized)
# Already created as X_train_std_df, X_val_std_df, X_test_std_df

# Model building functions
def build_fcnn(n_features, units1=128, units2=64, units3=32, 
               dropout_rate=0.3, learning_rate=0.001, l2_reg=0.001):
    """Build a fully connected neural network"""
    input_layer = Input(shape=(n_features,))
    
    # Layer 1
    x = Dense(units1, activation='relu', 
              kernel_regularizer=tf.keras.regularizers.l2(l2_reg))(input_layer)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    
    # Layer 2
    x = Dense(units2, activation='relu', 
              kernel_regularizer=tf.keras.regularizers.l2(l2_reg))(x)
    x = Dropout(dropout_rate / 2)(x)
    
    # Layer 3 (Optional)
    if units3 > 0:
        x = Dense(units3, activation='relu')(x)
    
    # Output layer
    output_layer = Dense(1, activation='linear')(x)
    
    model = Model(inputs=input_layer, outputs=output_layer)
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )
    return model

def build_bilstm(n_features, lstm_units=64, dense_units=64, 
                 dropout_rate=0.3, learning_rate=0.001, recurrent_dropout=0.1):
    """Build a bidirectional LSTM model"""
    # Input shape: (batch_size, timesteps, features)
    input_layer = Input(shape=(None, n_features))
    x = Masking(mask_value=0.0)(input_layer)
    
    # Bidirectional LSTM layer
    x = Bidirectional(LSTM(lstm_units, return_sequences=False, 
                          recurrent_dropout=recurrent_dropout))(x)
    x = Dropout(dropout_rate)(x)
    
    # Optional Dense layer
    if dense_units > 0:
        x = Dense(dense_units, activation='relu')(x)
        x = Dropout(dropout_rate)(x)
    
    output_layer = Dense(1, activation='linear')(x)
    
    model = Model(inputs=input_layer, outputs=output_layer)
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )
    return model

# Prepare data for BiLSTM
sequence_builder = SequenceBuilder(sequence_length=5)
X_train_seq, y_train_seq, groups_train_seq = sequence_builder.fit_transform(
    pd.concat([X_train_std_df, y_train], axis=1))
X_val_seq, y_val_seq, groups_val_seq = sequence_builder.transform(
    pd.concat([X_val_std_df, y_val], axis=1))
X_test_seq, y_test_seq, groups_test_seq = sequence_builder.transform(
    pd.concat([X_test_std_df, y_test], axis=1))

# Create and train FCNN model
print("Training FCNN model...")
fcnn_model = build_fcnn(n_features=X_train_std_df.shape[1])
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_fcnn = fcnn_model.fit(
    X_train_std, y_train,
    validation_data=(X_val_std, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

# Create and train BiLSTM model
print("Training BiLSTM model...")
bilstm_model = build_bilstm(n_features=X_train_std_df.shape[1])
history_bilstm = bilstm_model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluate models
def evaluate_model(model, X, y, model_type='fcnn'):
    """Evaluate model and return metrics"""
    if model_type == 'bilstm':
        # For BiLSTM, we need to prepare sequences
        seq_builder = SequenceBuilder(sequence_length=5)
        X_seq, y_seq, _ = seq_builder.transform(pd.concat([X, y], axis=1))
        y_pred = model.predict(X_seq, verbose=0).flatten()
        y_true = y_seq
    else:
        y_pred = model.predict(X, verbose=0).flatten()
        y_true = y.values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return rmse, mae, r2, y_pred

# Evaluate on test set
fcnn_rmse, fcnn_mae, fcnn_r2, fcnn_pred = evaluate_model(
    fcnn_model, X_test_std_df, y_test, 'fcnn')
bilstm_rmse, bilstm_mae, bilstm_r2, bilstm_pred = evaluate_model(
    bilstm_model, X_test_std_df, y_test, 'bilstm')

print("\n=== Model Performance on Test Set ===")
print(f"FCNN - RMSE: {fcnn_rmse:.4f}, MAE: {fcnn_mae:.4f}, R²: {fcnn_r2:.4f}")
print(f"BiLSTM - RMSE: {bilstm_rmse:.4f}, MAE: {bilstm_mae:.4f}, R²: {bilstm_r2:.4f}")

# Visualization of results
plt.figure(figsize=(12, 5))

# Plot predictions vs actual values
plt.subplot(1, 2, 1)
plt.scatter(y_test, fcnn_pred, alpha=0.5, label='FCNN')
plt.scatter(y_test, bilstm_pred, alpha=0.5, label='BiLSTM')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Predictions vs Actual Values')
plt.legend()

# Plot model performance comparison
plt.subplot(1, 2, 2)
metrics = ['RMSE', 'MAE', 'R²']
fcnn_scores = [fcnn_rmse, fcnn_mae, fcnn_r2]
bilstm_scores = [bilstm_rmse, bilstm_mae, bilstm_r2]

x = np.arange(len(metrics))
width = 0.35

plt.bar(x - width/2, fcnn_scores, width, label='FCNN')
plt.bar(x + width/2, bilstm_scores, width, label='BiLSTM')
plt.xlabel('Metrics')
plt.ylabel('Scores')
plt.title('Model Performance Comparison')
plt.xticks(x, metrics)
plt.legend()

plt.tight_layout()
plt.show()

# Residual analysis
fcnn_residuals = y_test - fcnn_pred
bilstm_residuals = y_test - bilstm_pred

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(fcnn_pred, fcnn_residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('FCNN Residual Plot')

plt.subplot(1, 2, 2)
plt.scatter(bilstm_pred, bilstm_residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('BiLSTM Residual Plot')

plt.tight_layout()
plt.show()

# Learning curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_fcnn.history['loss'], label='Training Loss')
plt.plot(history_fcnn.history['val_loss'], label='Validation Loss')
plt.title('FCNN Learning Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_bilstm.history['loss'], label='Training Loss')
plt.plot(history_bilstm.history['val_loss'], label='Validation Loss')
plt.title('BiLSTM Learning Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# Feature importance analysis (for FCNN)
# Get weights from the first layer
weights = fcnn_model.layers[1].get_weights()[0]
feature_importance = np.mean(np.abs(weights), axis=1)
feature_names = X_train_std_df.columns

# Sort features by importance
sorted_idx = np.argsort(feature_importance)[-10:]  # Top 10 features

plt.figure(figsize=(10, 6))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx])
plt.yticks(range(len(sorted_idx)), [feature_names[i] for i in sorted_idx])
plt.xlabel('Feature Importance (Average Absolute Weight)')
plt.title('Top 10 Important Features (FCNN)')
plt.tight_layout()
plt.show()

# Per-subject analysis
test_subjects = X_test['subject#'].unique()
subject_rmse_fcnn = []
subject_rmse_bilstm = []

for subject in test_subjects:
    subject_mask = X_test['subject#'] == subject
    if subject_mask.sum() > 0:  # Ensure subject exists in test set
        subject_y_true = y_test[subject_mask]
        
        # Get FCNN predictions for this subject
        subject_X = X_test_std_df[subject_mask]
        subject_fcnn_pred = fcnn_model.predict(subject_X, verbose=0).flatten()
        subject_rmse_fcnn.append(np.sqrt(mean_squared_error(subject_y_true, subject_fcnn_pred)))
        
        # Get BiLSTM predictions for this subject
        # We need to create sequences for this subject
        subject_df = pd.concat([X_test_std_df[subject_mask], y_test[subject_mask]], axis=1)
        seq_builder = SequenceBuilder(sequence_length=5)
        X_seq, y_seq, _ = seq_builder.transform(subject_df)
        if len(X_seq) > 0:
            subject_bilstm_pred = bilstm_model.predict(X_seq, verbose=0).flatten()
            subject_rmse_bilstm.append(np.sqrt(mean_squared_error(y_seq, subject_bilstm_pred)))
        else:
            subject_rmse_bilstm.append(np.nan)

# Plot per-subject RMSE
plt.figure(figsize=(10, 6))
x = np.arange(len(test_subjects))
width = 0.35

plt.bar(x - width/2, subject_rmse_fcnn, width, label='FCNN')
plt.bar(x + width/2, subject_rmse_bilstm, width, label='BiLSTM')
plt.xlabel('Subject ID')
plt.ylabel('RMSE')
plt.title('Per-Subject RMSE Comparison')
plt.xticks(x, test_subjects, rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

print("\n=== Summary ===")
print("The implementation includes:")
print("1. Comprehensive EDA and data visualization")
print("2. Proper train/validation/test splitting by subject")
print("3. Missing value imputation using KNN")
print("4. Feature engineering (temporal features, rolling statistics)")
print("5. Feature selection based on correlation with target")
print("6. Four dataset variants as required")
print("7. FCNN and BiLSTM model implementation")
print("8. Model evaluation with multiple metrics (RMSE, MAE, R²)")
print("9. Detailed visualization of results and model performance")
print("10. Residual analysis and feature importance examination")